# 06. Sistemas Multi-Agente

**Nivel:** 🔴 Avanzado  
**Tiempo estimado:** 90 minutos  
**Prerequisitos:** [01-05: Todos los notebooks anteriores](01-intro-llm-agents.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Diseñar sistemas con múltiples agentes especializados
- Implementar patrones de colaboración entre agentes
- Construir sistemas AutoGPT-style con task decomposition
- Entender BabyAGI y arquitecturas de planificación
- Manejar comunicación y coordinación multi-agente
- Aplicar patrones: Debate, Hierarchical, Sequential

## 1. Motivación: ¿Por Qué Multi-Agente?

### El Problema: Tareas Complejas Requieren Especialización

**Tarea:** *"Crea una aplicación web completa con frontend, backend, tests y documentación"*

**Un solo agente generalista:**
- Sobrecargado cognitivamente
- Contexto mezclado
- Calidad inconsistente
- Difícil de debuggear

**Sistema multi-agente:**
```
┌─────────────────────────────────────────────┐
│  Manager Agent (Orchestrator)              │
└──────────────┬──────────────────────────────┘
               │
      ┌────────┼────────┬────────┬────────┐
      ↓        ↓        ↓        ↓        ↓
  Frontend  Backend  Database  Test    Docs
   Agent     Agent    Agent    Agent   Agent
```

### Ventajas Multi-Agente

| Aspecto | Single Agent | Multi-Agent |
|---------|--------------|-------------|
| **Especialización** | Generalista | Expertos específicos |
| **Escalabilidad** | Limitada | Parallelizable |
| **Mantenimiento** | Difícil | Modular |
| **Debugging** | Complejo | Por agente |
| **Calidad** | Variable | Alta por dominio |

### Pregunta Guía

**Al final responderemos:**
*¿Cómo diseñar sistemas donde múltiples agentes colaboran efectivamente para resolver tareas complejas?*

In [ ]:
# Instalación
# !pip install openai anthropic python-dotenv

import os
import json
from typing import List, Dict, Optional, Tuple, Protocol
from dataclasses import dataclass, field
from enum import Enum
from abc import ABC, abstractmethod

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False

from dotenv import load_dotenv
load_dotenv()

print("✅ Librerías importadas")

## 2. Patrones de Multi-Agente

### Patrón 1: Sequential (Chain)

```
Input → Agent 1 → Agent 2 → Agent 3 → Output

Ejemplo: Research → Write → Review → Edit
```

### Patrón 2: Hierarchical (Manager-Worker)

```
         Manager
            │
      ┌─────┼─────┬─────┐
      ↓     ↓     ↓     ↓
    Agent Agent Agent Agent
      1     2     3     4
```

### Patrón 3: Debate / Consensus

```
        Problem
           │
      ┌────┼────┐
      ↓    ↓    ↓
    Agent Agent Agent
      1     2     3
      │     │     │
      └─────┼─────┘
        Debate
           ↓
      Consensus
```

### Patrón 4: AutoGPT (Autonomous Loop)

```
Goal
 ↓
┌──────────────────┐
│ 1. Think         │
│ 2. Plan          │
│ 3. Execute       │
│ 4. Evaluate      │
└────────┬─────────┘
         │
    Goal achieved?
    Yes → End
    No → Loop
```

## 3. Implementación: Base Agent Framework

In [ ]:
class AgentRole(Enum):
    """Roles de agentes especializados"""
    MANAGER = "manager"
    RESEARCHER = "researcher"
    WRITER = "writer"
    REVIEWER = "reviewer"
    CODER = "coder"
    TESTER = "tester"

@dataclass
class AgentMessage:
    """Mensaje entre agentes"""
    sender: str
    receiver: str
    content: str
    metadata: Dict = field(default_factory=dict)

class BaseAgent(ABC):
    """
    Clase base para agentes especializados.
    """
    
    def __init__(self, name: str, role: AgentRole, system_prompt: str):
        self.name = name
        self.role = role
        self.system_prompt = system_prompt
        self.memory = []  # Historial de mensajes
    
    @abstractmethod
    def process(self, task: str, context: Dict = None) -> str:
        """Procesa una tarea"""
        pass
    
    def send_message(self, receiver: str, content: str, metadata: Dict = None) -> AgentMessage:
        """Envía mensaje a otro agente"""
        msg = AgentMessage(
            sender=self.name,
            receiver=receiver,
            content=content,
            metadata=metadata or {}
        )
        return msg
    
    def receive_message(self, message: AgentMessage):
        """Recibe mensaje de otro agente"""
        self.memory.append(message)

print("✅ BaseAgent definido")

In [ ]:
class SpecializedAgent(BaseAgent):
    """
    Agente especializado con LLM.
    """
    
    def __init__(self, name: str, role: AgentRole, system_prompt: str, llm_backend: str = "simulated"):
        super().__init__(name, role, system_prompt)
        self.llm_backend = llm_backend
        
        if llm_backend == "openai" and OPENAI_AVAILABLE:
            self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            self.model = "gpt-4-turbo-preview"
        else:
            self.client = None
    
    def process(self, task: str, context: Dict = None) -> str:
        """Procesa tarea usando LLM con system prompt especializado"""
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": task}
        ]
        
        # Agregar contexto si existe
        if context:
            context_str = json.dumps(context, indent=2)
            messages.insert(1, {
                "role": "system",
                "content": f"Contexto adicional:\n{context_str}"
            })
        
        # Llamar LLM
        if self.llm_backend == "openai" and self.client:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.7
            )
            return response.choices[0].message.content
        else:
            # Simulado
            return self._simulated_response(task)
    
    def _simulated_response(self, task: str) -> str:
        """Respuesta simulada basada en role"""
        responses = {
            AgentRole.RESEARCHER: f"[{self.name}] Investigué sobre: {task}. Encontré información relevante...",
            AgentRole.WRITER: f"[{self.name}] Escribí contenido sobre: {task}. Aquí está el borrador...",
            AgentRole.REVIEWER: f"[{self.name}] Revisé: {task}. Sugerencias de mejora: ...",
            AgentRole.CODER: f"[{self.name}] Implementé código para: {task}. ```python\n# code here\n```",
            AgentRole.TESTER: f"[{self.name}] Probé: {task}. Resultados: ✅ Todos los tests pasan"
        }
        return responses.get(self.role, f"[{self.name}] Procesé: {task}")

print("✅ SpecializedAgent definido")

## 4. Patrón Sequential: Pipeline de Agentes

In [ ]:
class SequentialAgentPipeline:
    """
    Pipeline secuencial: output de un agente → input del siguiente.
    """
    
    def __init__(self, agents: List[BaseAgent]):
        self.agents = agents
    
    def run(self, initial_task: str, verbose: bool = True) -> str:
        """Ejecuta pipeline secuencial"""
        current_output = initial_task
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"🔄 Sequential Pipeline")
            print(f"{'='*70}\n")
            print(f"📋 Task inicial: {initial_task}\n")
        
        for i, agent in enumerate(self.agents, 1):
            if verbose:
                print(f"\n--- Paso {i}: {agent.name} ({agent.role.value}) ---")
            
            # Procesar con agente actual
            current_output = agent.process(current_output)
            
            if verbose:
                print(f"Output: {current_output[:200]}..." if len(current_output) > 200 else f"Output: {current_output}")
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"✅ Pipeline completado")
        
        return current_output

# Crear pipeline de blog post
research_agent = SpecializedAgent(
    name="Researcher",
    role=AgentRole.RESEARCHER,
    system_prompt="Eres un investigador experto. Investiga temas a fondo y proporciona información precisa."
)

writer_agent = SpecializedAgent(
    name="Writer",
    role=AgentRole.WRITER,
    system_prompt="Eres un escritor profesional. Creas contenido claro, atractivo y bien estructurado."
)

reviewer_agent = SpecializedAgent(
    name="Reviewer",
    role=AgentRole.REVIEWER,
    system_prompt="Eres un editor crítico. Revisas contenido para mejorar claridad, gramática y coherencia."
)

# Ejecutar pipeline
pipeline = SequentialAgentPipeline([research_agent, writer_agent, reviewer_agent])
result = pipeline.run("Escribe un blog post sobre LLM Agents")

print(f"\n📝 Resultado final:\n{result}")

## 5. Patrón Hierarchical: Manager-Worker

In [ ]:
@dataclass
class Task:
    """Representa una subtarea"""
    description: str
    assigned_to: Optional[str] = None
    status: str = "pending"  # pending, in_progress, completed
    result: Optional[str] = None

class ManagerAgent(BaseAgent):
    """
    Manager que descompone tareas y coordina workers.
    """
    
    def __init__(self, name: str, workers: Dict[str, BaseAgent]):
        super().__init__(name, AgentRole.MANAGER, "Manager que coordina agentes")
        self.workers = workers
        self.tasks: List[Task] = []
    
    def process(self, task: str, context: Dict = None) -> str:
        """Descompone tarea y delega a workers"""
        # 1. Descomponer en subtareas
        subtasks = self._decompose_task(task)
        
        print(f"\n🎯 Manager descompuso en {len(subtasks)} subtareas:\n")
        for i, st in enumerate(subtasks, 1):
            print(f"  {i}. {st.description} → {st.assigned_to}")
        
        # 2. Ejecutar subtareas
        results = []
        for subtask in subtasks:
            print(f"\n--- Ejecutando: {subtask.description} ---")
            
            worker = self.workers.get(subtask.assigned_to)
            if worker:
                subtask.status = "in_progress"
                result = worker.process(subtask.description)
                subtask.result = result
                subtask.status = "completed"
                results.append(result)
                print(f"✅ Completado")
            else:
                print(f"❌ Worker no encontrado: {subtask.assigned_to}")
        
        # 3. Sintetizar resultados
        final_result = self._synthesize_results(task, results)
        return final_result
    
    def _decompose_task(self, task: str) -> List[Task]:
        """Descompone tarea en subtareas (simplificado)"""
        # En producción, usar LLM para descomposición inteligente
        if "aplicación web" in task.lower():
            return [
                Task("Diseñar arquitectura del sistema", assigned_to="researcher"),
                Task("Implementar backend API", assigned_to="coder"),
                Task("Crear tests unitarios", assigned_to="tester"),
                Task("Escribir documentación", assigned_to="writer")
            ]
        elif "blog post" in task.lower() or "artículo" in task.lower():
            return [
                Task("Investigar tema", assigned_to="researcher"),
                Task("Escribir borrador", assigned_to="writer"),
                Task("Revisar y editar", assigned_to="reviewer")
            ]
        else:
            return [Task(task, assigned_to="researcher")]
    
    def _synthesize_results(self, original_task: str, results: List[str]) -> str:
        """Sintetiza resultados de subtareas"""
        synthesis = f"Tarea completada: {original_task}\n\n"
        synthesis += "Resultados de subtareas:\n"
        for i, result in enumerate(results, 1):
            synthesis += f"\n{i}. {result}\n"
        return synthesis

# Crear sistema hierarchical
coder_agent = SpecializedAgent(
    name="Coder",
    role=AgentRole.CODER,
    system_prompt="Eres un desarrollador experto. Escribes código limpio y eficiente."
)

tester_agent = SpecializedAgent(
    name="Tester",
    role=AgentRole.TESTER,
    system_prompt="Eres un QA engineer. Escribes tests completos y encuentras bugs."
)

workers = {
    "researcher": research_agent,
    "writer": writer_agent,
    "reviewer": reviewer_agent,
    "coder": coder_agent,
    "tester": tester_agent
}

manager = ManagerAgent("ProjectManager", workers)

# Ejecutar
print(f"\n{'='*70}")
print(f"👔 Hierarchical System: Manager + Workers")
print(f"{'='*70}")

result = manager.process("Crea una aplicación web simple")
print(f"\n📊 Resultado final del manager:\n{result}")

## 6. Patrón AutoGPT: Autonomous Loop

In [ ]:
class AutoGPTAgent:
    """
    Agente autónomo estilo AutoGPT.
    
    Loop:
    1. Think: Razonar sobre el estado actual
    2. Plan: Decidir próximas acciones
    3. Execute: Ejecutar acciones
    4. Evaluate: Evaluar si se alcanzó el objetivo
    5. Repeat o Finish
    """
    
    def __init__(self, goal: str, max_iterations: int = 10):
        self.goal = goal
        self.max_iterations = max_iterations
        self.task_history = []
        self.completed_tasks = []
    
    def run(self, verbose: bool = True) -> str:
        """Ejecuta loop autónomo"""
        if verbose:
            print(f"\n{'='*70}")
            print(f"🤖 AutoGPT Agent")
            print(f"{'='*70}")
            print(f"\n🎯 Goal: {self.goal}\n")
        
        for iteration in range(self.max_iterations):
            if verbose:
                print(f"\n--- Iteration {iteration + 1} ---\n")
            
            # 1. Think
            thought = self._think()
            if verbose:
                print(f"💭 Thought: {thought}")
            
            # 2. Plan
            plan = self._plan()
            if verbose:
                print(f"📋 Plan: {plan}")
            
            # 3. Execute
            result = self._execute(plan)
            if verbose:
                print(f"⚡ Result: {result}")
            
            self.completed_tasks.append({
                "thought": thought,
                "plan": plan,
                "result": result
            })
            
            # 4. Evaluate
            is_done = self._evaluate()
            if verbose:
                print(f"✅ Evaluation: {'Goal achieved!' if is_done else 'Continue...'}")
            
            if is_done:
                if verbose:
                    print(f"\n🎉 Goal achieved in {iteration + 1} iterations!")
                return self._generate_final_report()
        
        if verbose:
            print(f"\n⚠️  Max iterations reached without achieving goal")
        
        return self._generate_final_report()
    
    def _think(self) -> str:
        """Razona sobre estado actual"""
        completed = len(self.completed_tasks)
        return f"He completado {completed} tareas. Necesito continuar hacia: {self.goal}"
    
    def _plan(self) -> str:
        """Decide próxima acción"""
        # En producción, usar LLM para planning
        if len(self.completed_tasks) == 0:
            return "Comenzar investigando el tema"
        elif len(self.completed_tasks) == 1:
            return "Crear outline del contenido"
        elif len(self.completed_tasks) == 2:
            return "Escribir borrador completo"
        else:
            return "Revisar y finalizar"
    
    def _execute(self, plan: str) -> str:
        """Ejecuta plan"""
        # Simulación de ejecución
        return f"Ejecutado: {plan}"
    
    def _evaluate(self) -> bool:
        """Evalúa si se alcanzó el objetivo"""
        # Criterio simple: después de 4 tareas, considerar completo
        return len(self.completed_tasks) >= 4
    
    def _generate_final_report(self) -> str:
        """Genera reporte final"""
        report = f"AutoGPT Report: {self.goal}\n\n"
        report += f"Tareas completadas ({len(self.completed_tasks)}):"        for i, task in enumerate(self.completed_tasks, 1):
            report += f"\n{i}. {task['plan']} → {task['result']}"
        return report

# Ejecutar AutoGPT
autogpt = AutoGPTAgent(
    goal="Crear un artículo sobre LLM Agents",
    max_iterations=10
)

final_report = autogpt.run(verbose=True)
print(f"\n{final_report}")

## 7. Ejercicios

### 🟢 Ejercicio 1: Debate Pattern

In [ ]:
def ejercicio_1_debate_agents():
    """
    Objetivo: Implementar sistema de debate entre agentes
    
    Idea:
    - 3 agentes con perspectivas diferentes
    - Cada uno propone solución
    - Debaten pros/contras
    - Convergen a solución consensuada
    
    Ejemplo:
    Question: "¿Cómo reducir costos de API en agentes?"
    - Agent 1: "Usar modelos más pequeños"
    - Agent 2: "Implementar caching agresivo"
    - Agent 3: "Optimizar prompts"
    → Consensus: Combinar las tres estrategias
    """
    # TODO: Tu código aquí
    pass

### 🟡 Ejercicio 2: BabyAGI Implementation

In [ ]:
def ejercicio_2_babyagi():
    """
    Objetivo: Implementar BabyAGI pattern
    
    BabyAGI:
    1. Task Creation Agent: Genera nuevas tareas
    2. Prioritization Agent: Prioriza tareas pendientes
    3. Execution Agent: Ejecuta tarea top-priority
    4. Loop
    
    Instrucciones:
    1. Implementa cada agente
    2. Mantén task queue con prioridades
    3. Loop hasta que task queue esté vacío o max iterations
    """
    # TODO: Tu código aquí
    pass

### 🔴 Ejercicio 3: Multi-Agent Code Generation

In [ ]:
def ejercicio_3_code_generation_team():
    """
    Objetivo: Sistema multi-agente para generación de código
    
    Agentes:
    - Architect: Diseña estructura del código
    - Coder: Implementa funcionalidades
    - Reviewer: Code review y sugerencias
    - Tester: Escribe y ejecuta tests
    - Documenter: Genera documentación
    
    Tarea ejemplo:
    "Crea una clase Python para manejo de tareas con prioridad"
    
    Output esperado:
    - Código funcional
    - Tests que pasen
    - Documentación completa
    """
    # TODO: Tu código aquí
    pass

## 8. Resumen y Recursos

### 📚 Resumen

- **Multi-Agent Systems**: Especialización > Generalización
- **Patrones principales**:
  - Sequential: Pipeline lineal
  - Hierarchical: Manager + Workers
  - Debate: Múltiples perspectivas → Consensus
  - AutoGPT: Loop autónomo con evaluación
- **Ventajas**: Modularidad, escalabilidad, especialización
- **Desafíos**: Coordinación, comunicación, debugging

### 🔗 Recursos

#### 📄 Papers

1. **"AutoGPT: An Autonomous GPT-4 Experiment"**
2. **"BabyAGI: Task-driven Autonomous Agent"**
3. **"Communicative Agents for Software Development"** (ChatDev)

#### 💻 Proyectos Open Source

- **AutoGPT**: [GitHub](https://github.com/Significant-Gravitas/AutoGPT)
- **BabyAGI**: [GitHub](https://github.com/yoheinakajima/babyagi)
- **MetaGPT**: [GitHub](https://github.com/geekan/MetaGPT)
- **ChatDev**: [GitHub](https://github.com/OpenBMB/ChatDev)

### 🎓 Próximos Pasos

Has completado la ruta de LLM Agents! Ahora puedes:

1. **Practicar**: Implementa proyectos reales con agentes
2. **Profundizar**: Estudia papers recientes de agentic AI
3. **Contribuir**: Open-source en frameworks de agentes
4. **Explorar**: Otras rutas del repositorio

---

<div align="center">

### Respuesta a la Pregunta Guía

*¿Cómo diseñar sistemas multi-agente efectivos?*

**Claves:**
1. **Especialización**: Cada agente un rol específico
2. **Comunicación clara**: Protocolos definidos
3. **Coordinación**: Manager o protocol de consenso
4. **Modularidad**: Agentes independientes y reemplazables
5. **Evaluación**: Métricas de éxito por agente y sistema

Multi-agente es el futuro de AI systems complejos.

---

**¡Felicitaciones por completar la Ruta 3: LLM Agents!** 🎉

[← Volver al README de la Ruta](README.md) | [← Volver al README Principal](../../README.md)

</div>